CNNs
===

**Predict fear**

What is needed?

- Fear ratings (`fear_ratings.csv`)

## Imports

In [ ]:
import json
import os
import pickle
import random
import shutil
from glob import glob

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import timm
import torch
import torch.nn as nn
import torch.optim as optim
from cnn_utils import (
    OutputLayers,
    PredictionData,
    StimulusData,
    get_criterion,
    get_data_transforms,
    get_model_filepath,
    load_model,
    predict,
    set_parameter_requires_grad,
    train_model,
)
from skimage import io, transform
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset, TensorDataset
from torchvision import models, transforms

ARCHITECTURE = "resnet50"  # "resnet152" "resnet50" "inception_resnet", "resnet50d" "resnext101_32x8d", "densenet121"
CRITERION = "loss"  # optimize for 'loss' or 'acc'
ENSEMBLE_SIZE = 10  # number of models
LR_SCHEDULER_FACTOR = 0.8
LR_SCHEDULER_PATIENCE = 6
BATCH_SIZE_IM = 120  # Batch size for intermediate training
BATCH_SIZE_FT = 60
N_EPOCHS = 500  # max for training
EARLY_STOP_TOLERANCE = 30
DROPOUT = 0.5  # used to create layers for the intermediate model
N_DENSE = 256
N_LAYERS = 2
VAL_SIZE_REL = 0.20
FEATURE_EXTRACT_IM = True  # When False, we train the whole model,
FEATURE_EXTRACT_FT = False  # when True we only train the new layer
INPUT_SIZE = 224
FINAL_LAYER_NAME = "fc"
LR_IM = 6e-2  # alt: 5e-4, 10 ** (-2.2200654426745987)
IM_OPTIMIZER = "adam"
LR_FT = 5e-5  # alt: 3e-3, 1.5e-3,
FT_OPTIMIZER = "adam"
LOSS_TYPE = "L2"  # L1, L2, smooth_L1, huber, weighted_L2


def create_missing_data_dirs(directories):
    for directory in directories:
        if not os.path.isdir(directory):
            os.makedirs(directory)


if os.path.exists("data_dir"):
    with open("data_dir") as fo:
        DATA_DIR = fo.readlines()[0]
else:
    DATA_DIR = os.path.join(
        os.path.expanduser("~"), "Dropbox", "data_export", "mds-cnn-analysis"
    )
print("DATA_DIR:", DATA_DIR)
EXPORT_PATH_PLOTS = os.path.join(DATA_DIR, "plots")
EXPORT_PATH_DATA = os.path.join(DATA_DIR, "data")
IMG = os.path.join(DATA_DIR, "stimuli")
IMG_GLOB = sorted(glob(os.path.join(IMG, "*.jpg")))

TARGET_VAR = "fear_mean"  # "fear_std"

CHECKPOINTS = os.path.join(
    DATA_DIR,
    "CNN_checkpoints",
    TARGET_VAR,
    # IM_OPTIMIZER + "_" + FT_OPTIMIZER + "_" + LOSS_TYPE,
)
if not os.path.isdir(CHECKPOINTS):
    os.makedirs(CHECKPOINTS)
df_fear = pd.read_csv(os.path.join(EXPORT_PATH_DATA, "fear_ratings.csv"))

create_missing_data_dirs([DATA_DIR, CHECKPOINTS, EXPORT_PATH_PLOTS, EXPORT_PATH_DATA])

n_splits = 5

create_missing_data_dirs([DATA_DIR, CHECKPOINTS, EXPORT_PATH_PLOTS, EXPORT_PATH_DATA])

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device is", device)

## Prepare Data

### Load Train-Test Splits

Here we check which images were actually rated and then only include those ones.

In [ ]:
df_fear

In [ ]:
missing_idx = []

for i, file in enumerate(IMG_GLOB):
    if not file[-10:-4] in df_fear.image.values:
        print("Missing:", "index", i, "stimulus", file[-10:-4])
        missing_idx.append(i)

Prepare data

In [ ]:
with open(os.path.join(EXPORT_PATH_DATA, "5fold_train_idx.json")) as fo:
    five_fold_train_idx = json.load(fo)
with open(os.path.join(EXPORT_PATH_DATA, "5fold_test_idx.json")) as fo:
    five_fold_test_idx = json.load(fo)

emb_train_all_splits = []
emb_test_all_splits = []
image_paths_train_all_splits = []
image_paths_test_all_splits = []
image_train_dir_all_splits = []

for split in range(n_splits):
    train_idx = [idx for idx in five_fold_train_idx[split] if idx not in missing_idx]
    test_idx = [idx for idx in five_fold_test_idx[split] if idx not in missing_idx]

    np.savetxt(
        os.path.join(EXPORT_PATH_DATA, f"train_fear_idx_split_{split+1}.txt"), train_idx
    )
    np.savetxt(
        os.path.join(EXPORT_PATH_DATA, f"test_fear_idx_split_{split+1}.txt"), test_idx
    )

    img_dir_train = os.path.join(
        DATA_DIR, "stimuli_split", "fear", f"split_{split+1}", "img_train"
    )
    img_dir_test = os.path.join(
        DATA_DIR, "stimuli_split", "fear", f"split_{split+1}", "img_test"
    )

    if os.path.exists(img_dir_train):
        shutil.rmtree(img_dir_train, ignore_errors=True)

    if os.path.exists(img_dir_test):
        shutil.rmtree(img_dir_test, ignore_errors=True)

    os.makedirs(img_dir_train, exist_ok=True)
    os.makedirs(img_dir_test, exist_ok=True)

    for idx in train_idx:
        shutil.copy(IMG_GLOB[idx], img_dir_train)

    for idx in test_idx:
        shutil.copy(IMG_GLOB[idx], img_dir_test)

    image_paths_train = sorted(
        glob(img_dir_train + "/*.jpg") + glob(img_dir_train + "/*.png")
    )
    image_paths_test = sorted(
        glob(img_dir_test + "/*.jpg") + glob(img_dir_test + "/*.png")
    )

    train_images = pd.Series(image_paths_train, name="image").str[-10:-4]
    test_images = pd.Series(image_paths_test, name="image").str[-10:-4]

    emb_train = df_fear.merge(train_images, left_on="image", right_on="image")[
        TARGET_VAR
    ].values
    emb_test = df_fear.merge(test_images, left_on="image", right_on="image")[
        TARGET_VAR
    ].values
    emb_train = emb_train.reshape(emb_train.size, 1)

    print(f"\nSplit {split+1}:")
    print("emb_train.shape:", emb_train.shape, "emb_test.shape", emb_test.shape)

    emb_train_all_splits.append(emb_train)
    emb_test_all_splits.append(emb_test)
    image_paths_train_all_splits.append(image_paths_train)
    image_paths_test_all_splits.append(image_paths_test)
    image_train_dir_all_splits.append(img_dir_train)

### Transforms

In [ ]:
data_transforms = get_data_transforms(INPUT_SIZE)

### Create Dataset and Dataloaders

In [ ]:
train_dataset_all_splits = []
val_dataset_all_splits = []
test_dataset_all_splits = []
df_test_all_splits = []

for split in range(n_splits):
    emb_train = emb_train_all_splits[split]
    emb_test = emb_test_all_splits[split]
    img_train_dir = image_train_dir_all_splits[split]
    n_dim = emb_train.shape[1]
    img_train_dir = image_train_dir_all_splits[split]
    image_paths_train = image_paths_train_all_splits[split]
    image_paths_test = image_paths_test_all_splits[split]

    df = (
        pd.DataFrame(emb_train, columns=[str(dim) for dim in range(1, n_dim + 1)])
        .reset_index()
        .rename(columns={"index": "image_name"})
    )

    df["image_name"] = [os.path.basename(p) for p in image_paths_train]

    df_test = (
        pd.DataFrame(emb_test, columns=[str(dim) for dim in range(1, n_dim + 1)])
        .reset_index()
        .rename(columns={"index": "image_name"})
    )

    df_test["image_name"] = [os.path.basename(p) for p in image_paths_test]

    categories = len(df) * [0]

    # split train set again: train vs validate
    val_size_abs = round(len(image_paths_train) * VAL_SIZE_REL)
    train, val = train_test_split(
        df, test_size=val_size_abs, stratify=categories, random_state=0
    )

    # create image datasets
    train_dataset = StimulusData(
        train.reset_index(drop=True),
        root_dir=img_train_dir,
        device=device,
        transform=data_transforms["train"],
    )
    val_dataset = StimulusData(
        val.reset_index(drop=True),
        root_dir=img_train_dir,
        device=device,
        transform=data_transforms["val"],
    )
    test_dataset = PredictionData(image_paths_test, device)

    # append to all_splits lists
    train_dataset_all_splits.append(train_dataset)
    val_dataset_all_splits.append(val_dataset)
    test_dataset_all_splits.append(test_dataset)
    df_test_all_splits.append(df_test)

## Training

In [ ]:
for i in range(n_splits):
    print(
        f"""Cross-Validation Split: {i+1}/{len(emb_train_all_splits)}\n\n""",
        f"""Architecture: {ARCHITECTURE},
          Train goal: optimizing {CRITERION},
          Optimizer: IM: {IM_OPTIMIZER} (lr={LR_IM}), FT: {FT_OPTIMIZER} (lr={LR_FT})\n""",
    )

    checkpoints = os.path.join(CHECKPOINTS, f"split_{i+1}")
    create_missing_data_dirs([checkpoints])

    for e in range(1, ENSEMBLE_SIZE + 1):
        # check if model for this epoch already exists
        intermediate_available = os.path.isfile(
            get_model_filepath("intermediate", e, ARCHITECTURE, checkpoints)
        )

        if intermediate_available == False:
            print("intermediate stage")

            # Intermediate model
            if ARCHITECTURE == "resnet50":
                model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
            if ARCHITECTURE == "resnet50d":
                model = timm.create_model(
                    "resnet50d",
                    pretrained=False,
                    pretrained_cfg=models.ResNet50_Weights.DEFAULT,
                )
            elif ARCHITECTURE == "densenet121":
                model = models.densenet121(weights=models.DenseNet121_Weights.DEFAULT)
            elif ARCHITECTURE == "resnext101_32x8d":
                model = models.resnext101_32x8d(
                    weights=models.ResNeXt101_32X8D_Weights.DEFAULT
                )
            elif ARCHITECTURE == "inception_resnet":
                model = timm.create_model("inception_resnet_v2", pretrained=True)

            set_parameter_requires_grad(model, FEATURE_EXTRACT_IM)
            n_ftrs = model.get_submodule(FINAL_LAYER_NAME).in_features
            new_layers = OutputLayers(N_LAYERS, n_ftrs, N_DENSE, DROPOUT, n_dim)
            model.add_module(FINAL_LAYER_NAME, new_layers)  # replace last layer

            # Send the model to GPU
            model = model.to(device)

            # create datalaoders with specific batch size
            train_loader = DataLoader(
                train_dataset_all_splits[i], batch_size=BATCH_SIZE_IM
            )
            val_loader = DataLoader(val_dataset_all_splits[i], batch_size=BATCH_SIZE_IM)
            dataloaders_dict = {"train": train_loader, "val": val_loader}

            # Create Optimizer and define params to update
            params_to_update = model.parameters()
            if FEATURE_EXTRACT_IM:
                params_to_update = []
                for name, param in model.named_parameters():
                    if param.requires_grad == True:
                        params_to_update.append(param)

            else:
                for name, param in model.named_parameters():
                    if param.requires_grad == True:
                        ...

            # Instantiate optimizer for intermediate model
            if IM_OPTIMIZER == "sgd":
                optimizer = optim.SGD(params_to_update, lr=LR_IM, momentum=0.9)
            elif IM_OPTIMIZER == "adagrad":
                optimizer = optim.Adagrad(params_to_update, lr=LR_IM)
            elif IM_OPTIMIZER == "adadelta":
                optimizer = optim.Adadelta(params_to_update, lr=LR_IM)
            elif IM_OPTIMIZER == "adam":
                optimizer = optim.Adam(params_to_update, lr=LR_IM)

            # Setup the loss fxn
            criterion = get_criterion(LOSS_TYPE)

            # Initial training and evaluate
            model, hist_acc, hist_loss, hist_acc_train, hist_loss_train, lrs = (
                train_model(
                    model,
                    dataloaders_dict,
                    criterion,
                    optimizer,
                    n_dim,
                    device,
                    LR_SCHEDULER_PATIENCE,
                    LR_SCHEDULER_FACTOR,
                    num_epochs=N_EPOCHS,
                    early_stop_tolerance=EARLY_STOP_TOLERANCE,
                    crit=CRITERION,
                )
            )

            # Plot learning curve
            if CRITERION == "loss":
                hist = hist_loss
                hist_train = hist_loss_train
                hist_best = np.min(hist)
                hist_best_train = np.min(hist_train)
            elif CRITERION == "acc":
                hist = hist_acc
                hist_train = hist_acc_train
                hist_best = np.max(hist)
                hist_best_train = np.max(hist_train)

            plt.plot(
                hist_train,
                label=f"train (best: {hist_best_train:.3f})",
                color="gray",
                alpha=0.8,
            )
            plt.plot(hist, label=f"val (best: {hist_best:.3f})", color="salmon")
            plt.title(f"""Intermediate model {str(e).rjust(2, '0')}, split {i+1}""")
            plt.ylabel(CRITERION)
            plt.xlabel("Epoch")
            plt.legend()
            plt.show()

            # Save intermediate model
            torch.save(
                model.state_dict(),
                get_model_filepath("intermediate", e, ARCHITECTURE, checkpoints),
            )
            del model

        # check if fine-tuned model for this epoch already exists
        finetuned_available = os.path.isfile(
            get_model_filepath("final", e, ARCHITECTURE, checkpoints)
        )

        if not finetuned_available:

            model = load_model(
                "intermediate",
                e,
                FEATURE_EXTRACT_FT,
                FINAL_LAYER_NAME,
                N_LAYERS,
                N_DENSE,
                DROPOUT,
                checkpoints,
                device,
                n_dim,
                ARCHITECTURE,
            )

            # fine-tuning

            print("fine-tuning stage")

            # create dataloaders with specific batch size
            train_loader = DataLoader(
                train_dataset_all_splits[i], batch_size=BATCH_SIZE_FT
            )
            val_loader = DataLoader(val_dataset_all_splits[i], batch_size=BATCH_SIZE_FT)
            dataloaders_dict = {"train": train_loader, "val": val_loader}

            # Create Optimizer and define params to update
            params_to_update = model.parameters()
            if FEATURE_EXTRACT_FT:
                params_to_update = []
                for name, param in model.named_parameters():
                    if param.requires_grad == True:
                        params_to_update.append(param)
                        # print("\t",name)
            else:
                for name, param in model.named_parameters():
                    if param.requires_grad == True:
                        # print("\t",name)
                        ...

            # Instantiate optimizer for finetuning
            if FT_OPTIMIZER == "sgd":
                optimizer = optim.SGD(params_to_update, lr=LR_FT, momentum=0.9)
            elif FT_OPTIMIZER == "adagrad":
                optimizer = optim.Adagrad(params_to_update, lr=LR_FT)
            elif FT_OPTIMIZER == "adadelta":
                optimizer = optim.Adadelta(params_to_update, lr=LR_FT)
            elif FT_OPTIMIZER == "adam":
                optimizer = optim.Adam(params_to_update, lr=LR_FT)

            # Setup the loss fxn
            criterion = get_criterion(LOSS_TYPE)

            # Train and evaluate fine tuned model
            model, hist_acc, hist_loss, hist_acc_train, hist_loss_train, lrs = (
                train_model(
                    model,
                    dataloaders_dict,
                    criterion,
                    optimizer,
                    n_dim,
                    device,
                    LR_SCHEDULER_PATIENCE,
                    LR_SCHEDULER_FACTOR,
                    num_epochs=N_EPOCHS,
                    early_stop_tolerance=EARLY_STOP_TOLERANCE,
                    crit=CRITERION,
                )
            )

            # Plot learning curve
            if CRITERION == "loss":
                hist = hist_loss
                hist_train = hist_loss_train
                hist_best = np.min(hist)
                hist_best_train = np.min(hist_train)
            elif CRITERION == "acc":
                hist = hist_acc
                hist_train = hist_acc_train
                hist_best = np.max(hist)
                hist_best_train = np.max(hist_train)

            plt.plot(
                hist_train,
                label=f"train (best: {hist_best_train:.3f})",
                color="gray",
                alpha=0.8,
            )
            plt.plot(hist, label=f"val (best: {hist_best:.3f})", color="salmon")
            plt.title(f"""Final model {str(e).rjust(2, '0')}, split {i+1}""")
            plt.ylabel(CRITERION)
            plt.xlabel("Epoch")
            plt.legend()
            plt.show()

            # Save ensemble model
            torch.save(
                model.state_dict(),
                get_model_filepath("final", e, ARCHITECTURE, checkpoints),
            )

    print("Training run complete.")

## Prediction

### Function for creating summary DataFrame

In [ ]:
def create_summary_df(Y_pred_all_splits, Y_real_all_splits):
    """
    Create DataFrame with R² for all dimension, mean R², MSE and r
    for the ensemble models and all splits.
    """
    df_predictions = pd.DataFrame()

    i = 0

    for split in range(n_splits):
        Y_real = Y_real_all_splits[split]
        for model_type in Y_pred_all_splits.keys():
            Y_pred = Y_pred_all_splits[model_type][split]
            df_predictions.loc[i, "Split"] = str(split + 1)
            df_predictions.loc[i, "Model"] = model_type
            r2_raw = list(r2_score(Y_real, Y_pred, multioutput="raw_values"))

            if len(r2_raw) > 1:
                df_predictions.loc[
                    i, [f"R² Dim{dim}" for dim in range(1, 1 + len(r2_raw))]
                ] = r2_raw
                df_predictions.loc[i, "Mean R² (unif.)"] = r2_score(
                    Y_real, Y_pred, multioutput="uniform_average"
                )
                df_predictions.loc[i, "Mean R² (var.weig.)"] = r2_score(
                    Y_real, Y_pred, multioutput="variance_weighted"
                )
                df_predictions.loc[i, "Mean R² (flat vec.)"] = r2_score(
                    Y_real.reshape(Y_real.size),
                    Y_pred.reshape(Y_pred.size),
                    multioutput="uniform_average",
                )
            else:
                df_predictions.loc[i, "R²"] = r2_score(Y_real, Y_pred)
            df_predictions.loc[i, "r"] = np.corrcoef(
                Y_real.reshape(Y_real.size), Y_pred.reshape(Y_pred.size)
            )[0][1]
            df_predictions.loc[i, "MSE"] = mean_squared_error(Y_real, Y_pred)
            i += 1

    df_predictions_means = df_predictions.groupby(
        "Model", as_index=False, sort=False
    ).mean()
    df_predictions_means["Split"] = "Total"
    df_predictions = pd.concat(
        [df_predictions, df_predictions_means], sort=False
    ).reset_index(drop=True)

    return df_predictions

### Run Prediction

In [ ]:
%%time

model_types = ["intermediate", "final"]

Y_pred_all_splits = dict()
Y_pred_sd_all_splits = dict()
Y_pred_detail_all_splits = dict()

for model_type in model_types:

    Y_pred_all_splits[model_type] = []
    Y_pred_sd_all_splits[model_type] = []
    Y_pred_detail_all_splits[model_type] = []

    for split in range(n_splits):
        checkpoints = os.path.join(CHECKPOINTS, f"split_{split+1}")
        test_loader = DataLoader(
            test_dataset_all_splits[split], batch_size=1, shuffle=False, num_workers=0
        )

        Y_pred_detail = []

        for e in range(1, ENSEMBLE_SIZE + 1):
            try:
                model = load_model(
                    model_type,
                    e,
                    FEATURE_EXTRACT_FT,
                    FINAL_LAYER_NAME,
                    N_LAYERS,
                    N_DENSE,
                    DROPOUT,
                    checkpoints,
                    device,
                    n_dim,
                    ARCHITECTURE,
                )
                Y_pred_detail.append(
                    predict(model, test_loader, device, unlabeled=True)
                )

                del model
            except Exception as e:
                print(e)

        Y_pred = np.mean(Y_pred_detail, 0)
        Y_pred_sd = np.std(Y_pred_detail, 0)
        Y_pred_detail = np.array(Y_pred_detail)

        Y_pred_all_splits[model_type].append(Y_pred)
        Y_pred_sd_all_splits[model_type].append(Y_pred_sd)
        Y_pred_detail_all_splits[model_type].append(Y_pred_detail)

### MSE, Correlation and R²


In [ ]:
# create object for real/observed value
Y_real_all_splits = []
for split in range(n_splits):
    df_test = df_test_all_splits[split]
    Y_real = df_test.iloc[:, 1:].values
    Y_real_all_splits.append(Y_real)


df_predictions = create_summary_df(Y_pred_all_splits, Y_real_all_splits)
df_predictions

### Detailed R² by Models and Dimensions

This is only informative if there is more than one dimension.

In [ ]:
show_individual_ensembles = True

df_r2_all_splits = []

df_r2_details = pd.DataFrame()

for split in range(n_splits):

    all_model_types = {}

    for model_type in Y_pred_all_splits.keys():

        Y_real = Y_real_all_splits[split]
        Y_pred_detail = Y_pred_detail_all_splits[model_type][split]
        df_r2 = pd.DataFrame(index=range(1, Y_real.shape[1] + 1))

        for e in range(len(Y_pred_detail)):
            r2 = r2_score(Y_real, Y_pred_detail[e], multioutput="raw_values")
            df_r2[f"Model {e+1}"] = r2

        df_r2.loc["All", :] = df_r2.mean(axis=0)
        df_r2["Best model"] = df_r2.columns[df_r2.values.argmax(axis=1)]

        df_r2 = df_r2.reset_index().rename(columns={"index": "Dim"})
        df_r2.insert(0, "Model", model_type)
        df_r2.insert(0, "Split", split + 1)
        df_r2_details = pd.concat([df_r2_details, df_r2], sort=False).reset_index(
            drop=True
        )

df_r2_details

### Save Predictions to File

In [ ]:
from datetime import datetime as dt

td = dt.today().strftime("%y%m%d_%H%M")

with open(
    os.path.join(EXPORT_PATH_DATA, f"""CNN_observed_{TARGET_VAR}_{td}.pkl"""), "wb"
) as fo:
    pickle.dump(Y_real_all_splits, fo)

with open(
    os.path.join(
        EXPORT_PATH_DATA, f"""CNN_predictions_{TARGET_VAR}_{ARCHITECTURE}_{td}.pkl"""
    ),
    "wb",
) as fo:
    pickle.dump(Y_pred_all_splits, fo)

with open(
    os.path.join(
        EXPORT_PATH_DATA,
        f"""CNN_predictions_details_{TARGET_VAR}_{ARCHITECTURE}_{td}.pkl""",
    ),
    "wb",
) as fo:
    pickle.dump(Y_pred_detail_all_splits, fo)

with open(
    os.path.join(
        EXPORT_PATH_DATA,
        f"""CNN_predictions_std_{TARGET_VAR}_{ARCHITECTURE}_{td}.pkl""",
    ),
    "wb",
) as fo:
    pickle.dump(Y_pred_sd_all_splits, fo)


print("Saved with timestamp:", td)

### Load Predictions from File

In [ ]:
# td_ = "231011_0326"
td_ = "240403_1352"  # latest
td_ = "240416_0017"  # fear_std
td_ = "240416_1533"

with open(
    os.path.join(EXPORT_PATH_DATA, f"""CNN_observed_{TARGET_VAR}_{td_}.pkl"""), "rb"
) as fo:
    Y_real_all_splits = pickle.load(fo)

with open(
    os.path.join(
        EXPORT_PATH_DATA, f"""CNN_predictions_{TARGET_VAR}_resnet50_{td_}.pkl"""
    ),
    "rb",
) as fo:
    Y_pred_all_splits = pickle.load(fo)

with open(
    os.path.join(
        EXPORT_PATH_DATA, f"""CNN_predictions_details_{TARGET_VAR}_resnet50_{td_}.pkl"""
    ),
    "rb",
) as fo:
    Y_pred_detail_all_splits = pickle.load(fo)

with open(
    os.path.join(
        EXPORT_PATH_DATA, f"""CNN_predictions_std_{TARGET_VAR}_resnet50_{td_}.pkl"""
    ),
    "rb",
) as fo:
    Y_pred_sd_all_splits = pickle.load(fo)

df_predictions = create_summary_df(Y_pred_all_splits, Y_real_all_splits)
df_predictions

### Correlations of Observed vs. Predicted Fear

#### Full Plot

Function

In [ ]:
import seaborn as sns
from matplotlib.ticker import MaxNLocator
from plot_utils import get_empty_ax, imscatter

tick_locator = MaxNLocator(
    integer=True, steps=[1, 10]
)  # to ensure steps of 10 on each plot


def get_complete_ax(ax, obs, pred, images, zoom, max_size, radius, title):
    fontsize_legend = 13
    fontsize_title = 16
    fontsize_labels = 13

    x = obs
    y = pred
    r2 = r2_score(x, y)

    ax.add_artist(
        plt.Line2D([0], [0], color="white", label=f"$R^2$ = {r2:,.2f}", linewidth=0)
    )

    for i, image in enumerate(images):
        imscatter(
            x[i],
            y[i],
            image,
            ax=ax,
            zoom=zoom,
            max_size=max_size,
            shape="rounded",
            radius=radius,
        )

    ax.set_ylabel("predicted", fontsize=fontsize_labels)
    ax.set_xlabel("observed", fontsize=fontsize_labels)
    ax.set_title(title, fontsize=fontsize_title)

    margin = 0.05
    margin_x = abs(margin * (x.max() - x.min()))
    margin_y = abs(margin * (y.max() - y.min()))
    min = np.min((x.min() - margin_x, y.min() - margin_y))
    max = np.max((x.max() + margin_x, y.max() + margin_y))
    ax.set_xlim(min, max)
    ax.set_ylim(min, max)

    ax.axline(
        (1, 1),
        slope=1,
        c="gray",
        alpha=0.8,
        linestyle="--",
    )

    ax.tick_params(
        axis="both",
        which="both",
        bottom=False,
        top=False,
        labelbottom=False,
        left=False,
        labelleft=False,
    )

    ax.legend(loc="lower right", frameon=False, fontsize=fontsize_legend)

    return ax


def plot_compare_preds_1d(
    obs, pred, images, fname, radius, max_size, zoom=0.18, dpi=100, title=None
):
    """
    Plots and saves figure with observed vs. predicted values to file
    """
    assert len(obs) == len(pred), "Arrays obs and pred need to have same length!"

    fig, ax = plt.subplots(
        1, 1, figsize=(5 * 1, 5 * 1), constrained_layout=False, facecolor="#fff"
    )

    ax = get_complete_ax(ax, obs, pred, images, zoom, max_size, radius, title=title)

    fig.subplots_adjust(
        left=0.07, bottom=0.05, right=0.995, top=0.96, wspace=0.13, hspace=0.13
    )

    plt.savefig(
        fname,
        dpi=dpi,
        metadata=None,
        facecolor="white",
        edgecolor="auto",
    )

    plt.show()

    return None

Loop to create plots for each splits

In [ ]:
#### Y_real_all_flat = []
Y_pred_all_flat = {}
img_test_all_flat_glob = []
Y_real_all_flat = []

for split in range(n_splits):
    print("\nSplit:", split + 1, "\n\n")
    Y_real = Y_real_all_splits[split]
    img_test_glob = image_paths_test_all_splits[split]
    Y_real_all_flat += list(Y_real)
    img_test_all_flat_glob += img_test_glob

    for model_type in Y_pred_all_splits.keys():
        Y_pred = Y_pred_all_splits[model_type][split]
        print("\nModel:", model_type, "\n\n")
        if model_type not in Y_pred_all_flat:
            Y_pred_all_flat[model_type] = []
        Y_pred_all_flat[model_type] += list(Y_pred)

        plot_compare_preds_1d(
            Y_real,
            Y_pred,
            img_test_glob,
            os.path.join(
                EXPORT_PATH_PLOTS,
                f"CNN_{TARGET_VAR}_{td_}_{model_type}_split_{split+1}.pdf",
            ),
            max_size=160,
            zoom=0.15,
            dpi=300,
            radius=30,
            title=f"{TARGET_VAR.capitalize().replace('_', ' ')} (direct prediction)",
        )


# merge plots into 1

print("\nAll splits in one plot\n")

for model_type in Y_pred_all_splits.keys():
    print("\nModel:", model_type, "\n\n")
    plot_compare_preds_1d(
        np.array(Y_real_all_flat),
        np.array(Y_pred_all_flat[model_type]),
        img_test_all_flat_glob,
        os.path.join(EXPORT_PATH_PLOTS, f"CNN_{TARGET_VAR}_{td_}_{model_type}.pdf"),
        max_size=160,
        zoom=0.10,
        dpi=300,
        radius=30,
        title=f"{TARGET_VAR.capitalize().replace('_', ' ')} (direct prediction)",
    )

<div class="alert alert-info">Slope with confidence intervals removed for better clarity. Uncomment if you want to plot it.</div>

#### Simple Plot

Function

In [ ]:
def get_complete_ax_reduced(ax, obs, pred, title=None):
    fontsize_legend = 16
    fontsize_title = 12
    fontsize_labels = 16

    # compute regression
    x = obs
    y = pred

    # slope, intercept, r, p, std_err = linregress(x, y)
    r2 = r2_score(x, y)
    # Regression and confidence interval
    sns.regplot(
        x=x,
        y=y,
        x_ci="ci",
        scatter=True,
        ax=ax,
        marker="s",
        label=f"$R²$ = {r2:,.2f}",
        line_kws=dict(lw=1.5, color="black"),
        scatter_kws=dict(color="white", alpha=0.3, edgecolor="black", lw=1.5, s=50),
        robust=True,
        n_boot=1_000,
    )

    ax.set_ylabel("predicted", fontsize=fontsize_labels)
    ax.set_xlabel("observed", fontsize=fontsize_labels)
    ax.set_title(title, fontsize=fontsize_title)

    ax.tick_params(
        axis="both",
        which="both",
        bottom=False,
        top=False,
        labelbottom=False,
        left=False,
        labelleft=False,
    )

    min = 0.94 * np.min([np.min(y), np.min(x)])
    max = 1.06 * np.max([np.max(y), np.max(x)])
    ax.set_xlim((min, max))
    ax.set_ylim((min, max))
    ax.legend(loc="lower right", frameon=False, fontsize=fontsize_legend)

    return ax


def plot_compare_preds_1d_reduced(obs, pred, fname, dpi=100, title=None):
    """
    Plots and saves minimalist figure with observed vs. predicted values to file
    """

    assert len(obs) == len(pred), "Arrays obs and pred need to have same length!"

    fig, ax = plt.subplots(
        1, 1, figsize=(5 * 1, 5 * 1), constrained_layout=False, facecolor="#fff"
    )

    ax = get_complete_ax_reduced(ax, obs, pred, title=title)

    fig.subplots_adjust(
        left=0.07, bottom=0.05, right=0.995, top=0.96, wspace=0.13, hspace=0.13
    )

    plt.savefig(
        fname,
        dpi=dpi,
        metadata=None,
        facecolor="white",
        edgecolor="auto",
    )
    plt.show()

    return None

Loop over all splits

In [ ]:
for split in range(n_splits):
    print("\nSplit:", split + 1, "\n\n")
    Y_real = Y_real_all_splits[split]
    for model_type in Y_pred_all_splits.keys():
        print("\nModel:", model_type, "\n\n")
        Y_pred = Y_pred_all_splits[model_type][split]

        plot_compare_preds_1d_reduced(
            Y_real,
            Y_pred,
            os.path.join(
                EXPORT_PATH_PLOTS,
                f"CNN_{TARGET_VAR}_{model_type}_{td_}_split_{split+1}_reduced.pdf",
            ),
            dpi=240,
        )

print("\nAll splits in one plot\n\n")

for model_type in Y_pred_all_splits.keys():
    print("\nModel:", model_type, "\n\n")

    plot_compare_preds_1d_reduced(
        Y_real_all_flat,
        Y_pred_all_flat[model_type],
        os.path.join(
            EXPORT_PATH_PLOTS,
            f"CNN_{TARGET_VAR}_{model_type}_{td_}_split_{i+1}_reduced.pdf",
        ),
        dpi=240,
    )

### Uncertainty

#### Variance

In [ ]:
model_type = "final"
split = 0

Y_pred_sd_all_splits[model_type][split]

#### Distribution

In [ ]:
Y_pred_all_flat = {}
Y_pred_detail_all_flat = {}
img_test_all_flat_glob = []
Y_real_all_flat = []

for split in range(n_splits):
    Y_real = Y_real_all_splits[split]
    img_test_glob = image_paths_test_all_splits[split]
    Y_real_all_flat += list(Y_real)
    img_test_all_flat_glob += img_test_glob

    for model_type in Y_pred_all_splits.keys():
        Y_pred = Y_pred_all_splits[model_type][split]
        Y_pred_detail = np.swapaxes(Y_pred_detail_all_splits[model_type][split], 0, 1)
        if model_type not in Y_pred_all_flat:
            Y_pred_all_flat[model_type] = []
        if model_type not in Y_pred_detail_all_flat:
            Y_pred_detail_all_flat[model_type] = []
        Y_pred_all_flat[model_type] += list(Y_pred)
        Y_pred_detail_all_flat[model_type] += list(Y_pred_detail)

In [ ]:
model_type = "final"

for i_img in range(len(Y_pred_all_flat[model_type])):

    fig, axs = plt.subplots(1, 2, figsize=(13, 5))

    min_val = Y_pred_detail_all_flat[model_type][i_img].min()
    max_val = Y_pred_detail_all_flat[model_type][i_img].max()
    mean_val = Y_pred_detail_all_flat[model_type][i_img].mean()
    std_val = Y_pred_detail_all_flat[model_type][i_img].std()
    real_val = Y_real_all_flat[i_img][0]

    axs[0].imshow(plt.imread(img_test_all_flat_glob[i_img]))
    axs[0].axis("off")
    axs[0].set_title(f"{os.path.basename(img_test_all_flat_glob[i_img])[:-4]}")

    # convert to int for plot
    axs[1].hist(Y_pred_detail_all_flat[model_type][i_img].round(0).astype(int))
    axs[1].axvline(
        real_val, label=f"Rating ({real_val:.2f})", color="red", linestyle="--"
    )
    axs[1].axvline(
        mean_val,
        label=f"Mean Prediction ({mean_val:.2f})",
        color="black",
        linestyle="--",
    )
    axs[1].axvline(
        mean_val - std_val,
        label=f"+- 1 SD ({std_val:.2f})",
        color="lightgray",
        linestyle="--",
    )
    axs[1].axvline(mean_val + std_val, color="lightgray", linestyle="--")
    axs[1].set_xlim((0, 100))
    # axs[1].set_title(f"M={mean_val:.2f}, SD={std_val:.2f} (min: {min_val:.2f}, max: {max_val:.2f})")
    axs[1].legend()
    plt.show()

Distribution of standard deviations

In [ ]:
all_stds = np.array(Y_pred_detail_all_flat[model_type]).std(axis=1)

plt.hist(all_stds, bins=70)
plt.axvline(
    all_stds.mean(), label=f"M={all_stds.mean():.2f}", color="black", linestyle="--"
)
plt.title(f"Distribution of standard deviations (N={len(all_stds)})")
plt.legend();

In [ ]:
import seaborn as sns

x = np.array(Y_real_all_flat).T[0]
y = all_stds
sns.regplot(x=x, y=y)
plt.ylabel("SD")
plt.xlabel("Rating")
plt.title(f"r={np.corrcoef(x, y)[0][1]:.2f}")

In [ ]:
x = Y_pred_all_flat[model_type]
y = all_stds
sns.regplot(x=x, y=y)
plt.ylabel("SD")
plt.xlabel("Mean prediction")
plt.title(f"r={np.corrcoef(x, y)[0][1]:.2f}")

<div class="alert alert-info">Higher variance with low-fear images.</div>

#### MC Dropout

In [ ]:
#
#
#

### Check for Correlations Between Dimensions

<div class="alert alert-danger">Below still not functional</div>

In [ ]:
from plot_utils import plot_corr_dims

for split in range(n_splits):
    print("\nSplit:", split + 1, "\n\n")
    Y_real = Y_real_all_splits[split]
    plot_corr_dims(Y_real, "Observed")
    plt.show()

In [ ]:
model_type = "final"

for split in range(n_splits):
    print("\nSplit:", split + 1, "\n\n")
    Y_pred = Y_pred_all_splits[model_type][split]
    plot_corr_dims(Y_pred, "Predicted")
    plt.show()

## Example predictions

In [ ]:
Y_real_all_splits[0][58]

In [ ]:
np.argmax(Y_pred_all_splits["final"][0])

In [ ]:
Y_pred_all_splits["final"][0][58]

In [ ]:
plt.imshow(plt.imread(image_paths_test_all_splits[0][58]))